# Homework 8.2 - Part III

## Fokker-Planck Equation

The state vector is $(X_t,L_t)$ with drift

$$
b_x(x,\ell)
=
a(-x+\ell)
$$

and

$$
b_\ell(x,\ell)
=
-\kappa
\frac{|x|}
{1+\eta|\ell|}.
$$

Only $X_t$ contains an independent Brownian increment.

Therefore the joint Fokker-Planck equation is

$$
\frac{\partial p}{\partial t}
=
-\frac{\partial}{\partial x}
\left[
a(-x+\ell)p
\right]
-
\frac{\partial}{\partial \ell}
\left[
-\kappa
\frac{|x|}
{1+\eta|\ell|}
p
\right]
+
\frac{\sigma^2}{2}
\frac{\partial^2p}{\partial x^2}.
$$

## Why there is no diffusion term in the L direction

There is no independent $dW_t$ term in the equation for $L_t$.

Therefore the diffusion matrix has nonzero variance only in the $X$ coordinate.

However, $L_t$ is still random because its drift depends on the random process $X_t$.

Thus:

- $L_t$ can be random;
- its sample paths have finite variation;
- randomness transmitted through a random drift is not the same as an independent Brownian diffusion term.

## Weak interpretation

The initial state is deterministic:

$$
(X_0,L_0)=(1,0).
$$

The initial joint law is therefore a point mass rather than a smooth classical density.

A safe formulation uses smooth test functions $\phi$.

The generator is

$$
A\phi
=
a(-x+\ell)\phi_x
-
\kappa
\frac{|x|}
{1+\eta|\ell|}
\phi_\ell
+
\frac{\sigma^2}{2}
\phi_{xx}.
$$

The weak identity is

$$
\frac{d}{dt}
E[
\phi(X_t,L_t)
]
=
E[
A\phi(X_t,L_t)
].
$$

The next experiment checks this generator identity numerically.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "hw8_2"

config = json.loads(
    (
        DATA_DIR
        / "hw8_2_configuration.json"
    ).read_text(
        encoding="utf-8"
    )
)

A = float(
    config["a"]
)

KAPPA = float(
    config["kappa"]
)

SIGMA = float(
    config["sigma"]
)

ETA = float(
    config["eta"]
)

X0 = float(
    config["x0"]
)

L0 = float(
    config["l0"]
)

DT = 0.002
T_EVAL = 2.0
N_PATHS = 100000
SEED = 63831

In [2]:
def F(l_value, x_value):
    return (
        np.abs(
            x_value
        )
        / (
            1.0
            + ETA
            * np.abs(
                l_value
            )
        )
    )


rng = np.random.default_rng(
    SEED
)

x = np.full(
    N_PATHS,
    X0,
    dtype=float,
)

l = np.full(
    N_PATHS,
    L0,
    dtype=float,
)

n_steps = int(
    round(
        T_EVAL / DT
    )
)

for _ in range(
    n_steps
):
    z = rng.normal(
        size=N_PATHS
    )

    x_old = x.copy()
    l_old = l.copy()

    x = (
        x_old
        + A
        * (
            -x_old
            + l_old
        )
        * DT
        + SIGMA
        * np.sqrt(
            DT
        )
        * z
    )

    l = (
        l_old
        - KAPPA
        * F(
            l_old,
            x_old,
        )
        * DT
    )


x_before = x.copy()
l_before = l.copy()

z = rng.normal(
    size=N_PATHS
)

x_after = (
    x_before
    + A
    * (
        -x_before
        + l_before
    )
    * DT
    + SIGMA
    * np.sqrt(
        DT
    )
    * z
)

l_after = (
    l_before
    - KAPPA
    * F(
        l_before,
        x_before,
    )
    * DT
)

## Test functions

We use

$$
\phi_1=x,
$$

$$
\phi_2=\ell,
$$

$$
\phi_3=x^2,
$$

and

$$
\phi_4=x\ell.
$$

Their generators are computed analytically and compared with the one-step Monte Carlo difference quotient.

In [3]:
f_value = F(
    l_before,
    x_before,
)

tests = []

# phi = x
lhs = (
    np.mean(
        x_after
    )
    - np.mean(
        x_before
    )
) / DT

rhs = np.mean(
    A
    * (
        -x_before
        + l_before
    )
)

tests.append({
    "test_function": "x",
    "finite_difference_time_derivative": float(
        lhs
    ),
    "mean_generator": float(
        rhs
    ),
    "absolute_difference": float(
        abs(
            lhs - rhs
        )
    ),
})

# phi = l
lhs = (
    np.mean(
        l_after
    )
    - np.mean(
        l_before
    )
) / DT

rhs = np.mean(
    -KAPPA
    * f_value
)

tests.append({
    "test_function": "l",
    "finite_difference_time_derivative": float(
        lhs
    ),
    "mean_generator": float(
        rhs
    ),
    "absolute_difference": float(
        abs(
            lhs - rhs
        )
    ),
})

# phi = x^2
lhs = (
    np.mean(
        x_after**2
    )
    - np.mean(
        x_before**2
    )
) / DT

rhs = np.mean(
    2.0
    * x_before
    * A
    * (
        -x_before
        + l_before
    )
    + SIGMA**2
)

tests.append({
    "test_function": "x^2",
    "finite_difference_time_derivative": float(
        lhs
    ),
    "mean_generator": float(
        rhs
    ),
    "absolute_difference": float(
        abs(
            lhs - rhs
        )
    ),
})

# phi = x*l
lhs = (
    np.mean(
        x_after
        * l_after
    )
    - np.mean(
        x_before
        * l_before
    )
) / DT

rhs = np.mean(
    l_before
    * A
    * (
        -x_before
        + l_before
    )
    + x_before
    * (
        -KAPPA
        * f_value
    )
)

tests.append({
    "test_function": "x*l",
    "finite_difference_time_derivative": float(
        lhs
    ),
    "mean_generator": float(
        rhs
    ),
    "absolute_difference": float(
        abs(
            lhs - rhs
        )
    ),
})


weak_validation = pd.DataFrame(
    tests
)

weak_validation.to_csv(
    DATA_DIR
    / "part3_weak_generator_validation.csv",
    index=False,
)

print(
    weak_validation.to_string(
        index=False
    )
)

test_function  finite_difference_time_derivative  mean_generator  absolute_difference
            x                          -0.193766       -0.200902         7.135176e-03
            l                          -0.016260       -0.016260         5.759282e-16
          x^2                          -0.132926       -0.132465         4.606210e-04
          x*l                           0.005444        0.005879         4.351441e-04


# Part III Conclusion

The Fokker-Planck equation has diffusion only in the $x$ direction because only $X_t$ receives an independent Brownian increment.

The random evolution of $L_t$ is transmitted through its dependence on $X_t$.

The Monte Carlo generator check provides a numerical consistency test of the weak formulation without pretending that a full two-dimensional PDE was solved on a spatial grid.